# Preparation layer

This notebook focuses only on document preparation:

- structure-aware segmentation
- table + narrative preservation
- provenance metadata
- noise cleanup


In [1]:
from __future__ import annotations

import hashlib
import json
import re
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean
from typing import Iterable

import pdfplumber
from pypdf import PdfReader


In [2]:
# =============================================================================
# Configuration


In [3]:
# =============================================================================

DOCUMENTS = {
    "UK": Path(
        "/Users/umair/CCO-GRO(Updated)/1 - Data Preparation Layer/PDFs Processing/raw_docs/Funding_Rules_(uk).pdf"
    ),
    "Canada": Path(
        "/Users/umair/CCO-GRO(Updated)/1 - Data Preparation Layer/PDFs Processing/raw_docs/policy_manual(canada).pdf"
    ),
    "Australia": Path(
        "/Users/umair/CCO-GRO(Updated)/1 - Data Preparation Layer/PDFs Processing/raw_docs/VSL Provider Manual(aus).pdf"
    ),
    "USA": Path(
        "/Users/umair/CCO-GRO(Updated)/1 - Data Preparation Layer/PDFs Processing/raw_docs/USA_fsa-handbook_2025-2026_application-and-verification-guide.pdf"
    ),
}

OUTPUT_DIR = Path("output/preparation_v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHS = r"January|February|March|April|May|June|July|August|September|October|November|December"


In [4]:
# =============================================================================
# Data Models


In [5]:
# =============================================================================


@dataclass
class OutlineEntry:
    title: str
    page: int
    depth: int
    source: str
    printed_page: str | None = None


@dataclass
class Chunk:
    jurisdiction: str
    source_file: str
    pdf_page: int
    printed_page: str | None
    section_path: list[str]
    local_heading: str | None
    paragraph_number: str | None
    content_type: str
    text: str
    source_anchor: str
    thresholds: list[str] = field(default_factory=list)
    temporal_cues: list[str] = field(default_factory=list)


@dataclass
class PreparedUnit:
    jurisdiction: str
    source_file: str
    unit_id: str
    unit_type: str
    pdf_page_start: int
    pdf_page_end: int
    printed_page_start: str | None
    printed_page_end: str | None
    section_path: list[str]
    section: str
    section_id: str
    local_heading: str | None
    paragraph_number: str | None
    source_anchor: str
    text: str
    component_chunk_ids: list[str]
    support_chunk_ids: list[str] = field(default_factory=list)
    thresholds: list[str] = field(default_factory=list)
    temporal_cues: list[str] = field(default_factory=list)


In [6]:
# =============================================================================
# Text Normalisation


In [7]:
# =============================================================================


_KNOWN_LEGIT_4_TOKENS = {
    "T4A", "T4As", "T4", "T5",   # Canadian tax slips
    "W4", "W-4", "K1", "K-1",    # US tax forms
}


def normalize_whitespace(text: str) -> str:
    """
    Whitespace + PDF extraction artifact cleanup.
    Substitutions are context-bound + whitelist-protected to prevent
    false positives on legitimate tokens (W9a, Section4, v4, G4, Form4a,
    T4A Canadian tax slip, W-4, Q1..Q99, case IDs, etc.).
    """
    text = (text or "").replace("\u00a0", " ")
    text = text.replace("\u2011", "-").replace("\u2010", "-")

    # `9 -> '` apostrophe inside a word: "person9s" -> "person's"
    text = re.sub(
        r"(?<=[a-z])9(?=(?:s|t|ll|ve|re|d|m)\b)",
        "'",
        text,
    )

    # `9 -> '` trailing possessive: "parents9 income" -> "parents' income"
    text = re.sub(
        r"(?<=[a-z]s)9(?=[\s.,;:!?\)]|$)",
        "'",
        text,
    )

    # `4 -> " - "` separator artifact. Single-letter context allowed on
    # each side, but a whitelist guard skips legitimate tax-form tokens
    # (T4A, T4, W4, K1, etc.) and short <upper>4<upper> patterns of the
    # same shape.
    def _sub_4(m: re.Match) -> str:
        s = m.string
        i = m.start()
        while i > 0 and s[i - 1].isalpha():
            i -= 1
        j = m.end()
        while j < len(s) and s[j].isalpha():
            j += 1
        full_token = s[i:j]
        if full_token in _KNOWN_LEGIT_4_TOKENS:
            return m.group(0)
        left_part = s[i:m.start()]
        right_part = s[m.end():j]
        # Shape guard: <single upper>4<one-or-two upper> looks like a form code
        if (len(left_part) == 1 and left_part.isupper()
                and right_part and right_part[0].isupper() and len(right_part) <= 2):
            return m.group(0)
        return " - "

    text = re.sub(r"(?<=[A-Za-z])4(?=[A-Za-z])", _sub_4, text)

    # `3 -> -` separator artifact: "Volume 4A 3 Record" -> "Volume 4A - Record",
    # "Volume 5 3 EDE" -> "Volume 5 - EDE". Requires the left context to end
    # in a digit (with optional trailing capital letter) and the right
    # context to be a Capitalized word or all-caps acronym. Protects normal
    # phrases like "page 3 of 5", "Example 5: A student", "Section 3 covers".
    text = re.sub(
        r"(?<=\d)\s+3\s+(?=[A-Z][A-Za-z])",
        " - ",
        text,
    )
    text = re.sub(
        r"(?<=\d[A-Z])\s+3\s+(?=[A-Z][A-Za-z])",
        " - ",
        text,
    )

    # Year-range collapse: "2023324" -> "2023-24". Fires only when trailing
    # 2 digits == (first_year_last_two + 1) % 100. Protects case IDs.
    def _year_range_sub(m: re.Match) -> str:
        full_year = m.group(1)
        trail = m.group(2)
        try:
            expected = (int(full_year[-2:]) + 1) % 100
            if int(trail) == expected:
                return f"{full_year}-{trail}"
        except ValueError:
            pass
        return m.group(0)

    text = re.sub(r"\b(20\d{2})3(\d{2})\b", _year_range_sub, text)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s+\n", "\n", text)
    return text.strip()


def _content_fingerprint(text: str) -> str:
    """
    Stable content fingerprint for deduplication. Uses full normalized
    text (not a truncated slug) to prevent silent data loss where two
    long paragraphs sharing the same first 64 chars were incorrectly
    treated as duplicates.
    """
    normalized = re.sub(r"\s+", " ", text.strip().lower())
    return hashlib.sha1(normalized.encode("utf-8")).hexdigest()


def canonical_heading(text: str) -> str:
    text = normalize_whitespace(text)
    text = re.sub(r"\.{4,}\s*\d+\s*$", "", text)
    text = re.sub(r"\s+\d{1,3}\s*$", "", text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return normalize_whitespace(text)


def clean_inline_furniture(text: str, jurisdiction: str) -> str:
    t = normalize_whitespace(text)
    if jurisdiction == "Canada":
        t = re.sub(
            r"StudentAid BC Policy Manual 2025-26\s+Effective:\s+June\s+11,\s+2025",
            "",
            t,
            flags=re.I,
        )
    elif jurisdiction == "Australia":
        t = re.sub(
            rf"VET Student Loans Provider Manual\s*[-–]\s*January\s*2026\s*\|\s*\d+",
            "",
            t,
            flags=re.I,
        )
    elif jurisdiction == "UK":
        t = re.sub(r"^\s*\d{1,3}\s*$", "", t)

    t = re.sub(r"\s+\|\s*$", "", t)
    t = re.sub(r"\s{2,}", " ", t)
    return t.strip(" |")


NOISE_PATTERNS = [
    r"crown\s+copyright",
    r"open\s+government\s+licence",
    r"nationalarchives\.gov\.uk",
    r"creative\s+commons",
    r"creativecommons\.org",
    r"commonwealth\s+coat\s+of\s+arms",
    r"this\s+publication\s+is\s+licensed",
]


def is_noise_text(text: str) -> bool:
    low = normalize_whitespace(text).lower()
    return any(re.search(pat, low, re.I) for pat in NOISE_PATTERNS)


REVERSED_WORDS = {
    "snoitutitsni",
    "cilbup",
    "gniniart",
    "revird",
    "gnireffo",
    "snoitutitsni",
    "lacigoloeht",
    "smargorp",
    "naidanac",
    "ecnivorp",
    "tneduts",
    "ecnatsissa",
    "laicnanif",
    "lanoitanretni",
}


# Common English palindromes that should NEVER trigger garbled detection.
# Words like "level", "civic", "refer" are valid English — not reversed artifacts.
PALINDROME_WHITELIST = {
    "level", "civic", "refer", "noon", "deed", "kayak", "radar",
    "racecar", "madam", "repaper", "rotator", "redder", "tenet",
    "stats", "sexes", "sagas",
}


def is_garbled_text(text: str) -> bool:
    """
    Detect table/OCR artifacts where words have been extracted reversed.

    Guards:
    - PALINDROME_WHITELIST: common valid English palindromes never trigger detection.
    - Single repeated palindrome word (e.g., "XXXX") in a table sample/placeholder
      does not trigger detection.
    - Only genuine reversed policy vocabulary triggers.
    """
    t = normalize_whitespace(text).lower()
    words = re.findall(r"[a-z]{5,}", t)
    alpha_tokens = re.findall(r"[a-z]+", t)

    # Single-char ratio check (spaced OCR artifacts: "s n o i t u t i t s n i")
    if len(alpha_tokens) >= 30:
        single_char_ratio = sum(1 for tok in alpha_tokens if len(tok) == 1) / len(alpha_tokens)
        if single_char_ratio >= 0.55:
            return True

    if not words:
        return False

    # Count genuine reversed hits — skip whitelisted palindromes
    reversed_hits = sum(
        1 for word in words
        if word in REVERSED_WORDS and word not in PALINDROME_WHITELIST
    )
    if text.lstrip().startswith("|") and reversed_hits >= 1:
        return True
    if reversed_hits >= 3:
        return True

    # Softer fallback: many long words whose reverse is a known policy term.
    # Guard: if ALL matching words are the same token, it is a placeholder
    # like "XXXX" in a sample form — not a garbled artifact.
    common_forward = {
        "institutions", "public", "training", "driver", "offering",
        "theological", "programs", "canadian", "student", "assistance",
        "financial", "international",
    }
    reverse_matches = [word for word in words if word[::-1] in common_forward]
    if len(reverse_matches) >= 3:
        # If all matching words are identical, it is a placeholder — skip
        if len(set(reverse_matches)) == 1:
            return False
        return True

    return False


def is_page_furniture(line: str, jurisdiction: str) -> bool:
    l = normalize_whitespace(line)
    if not l:
        return True
    if re.fullmatch(r"\d{1,4}", l):
        return True
    if re.fullmatch(r"(?:i|ii|iii|iv|v|vi|vii|viii|ix|x)", l, re.I):
        return True
    if jurisdiction == "Canada":
        return bool(
            re.fullmatch(
                r"StudentAid BC Policy Manual 2025-26\s+Effective:\s+June\s+11,\s+2025",
                l,
                re.I,
            )
        )
    if jurisdiction == "Australia":
        return bool(
            re.fullmatch(
                rf"VET Student Loans Provider Manual\s*[-–]\s*January\s*2026\s*\|\s*\d+",
                l,
                re.I,
            )
        )
    return False


def printed_page_from_lines(lines: list[str], jurisdiction: str, pdf_page: int) -> str | None:
    if jurisdiction == "Canada":
        for line in lines[:4]:
            m = re.fullmatch(r"(?:i|ii|iii|iv|v|vi|vii|viii|ix|x|\d{1,3})", line.strip(), re.I)
            if m:
                return m.group(0)
    if jurisdiction == "UK":
        for line in reversed(lines[-6:]):
            m = re.fullmatch(r"\d{1,3}", line.strip())
            if m:
                return m.group(0)
    if jurisdiction == "Australia":
        for line in reversed(lines[-6:]):
            m = re.search(r"\|\s*(\d{1,3})\s*$", line)
            if m:
                return m.group(1)
    return str(pdf_page)


In [8]:
# =============================================================================
# Structure Maps


In [9]:
# =============================================================================


def flatten_pdf_outline(reader: PdfReader, outline: Iterable | None = None, depth: int = 0) -> list[OutlineEntry]:
    if outline is None:
        outline = reader.outline
    rows: list[OutlineEntry] = []
    for item in outline or []:
        if isinstance(item, list):
            rows.extend(flatten_pdf_outline(reader, item, depth + 1))
            continue
        title = getattr(item, "title", None)
        if not title:
            continue
        try:
            page = reader.get_destination_page_number(item) + 1
        except Exception:
            continue
        rows.append(OutlineEntry(title=normalize_whitespace(title), page=page, depth=depth, source="bookmark"))
    return rows


def parse_australia_toc(pdf_path: Path) -> list[OutlineEntry]:
    entries: list[OutlineEntry] = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        text = "\n".join((pdf.pages[i].extract_text() or "") for i in range(1, min(4, len(pdf.pages))))

    for raw_line in text.splitlines():
        line = normalize_whitespace(raw_line)
        if not line or "........" not in line:
            continue
        m = re.match(r"^([A-Z]|\d+(?:\.\d+)?)\.\s+(.+?)\s*\.{4,}\s*(\d{1,3})$", line)
        if not m:
            continue
        number, title, printed_page = m.groups()
        if number.isdigit():
            depth = 0
            full_title = f"{number}. {title}"
        elif re.match(r"\d+\.\d+", number):
            depth = 1
            full_title = f"{number} {title}"
        else:
            depth = 0
            full_title = f"{number}. {title}"
        entries.append(
            OutlineEntry(
                title=normalize_whitespace(full_title),
                page=int(printed_page),
                depth=depth,
                source="printed_toc",
                printed_page=printed_page,
            )
        )
    return entries


def build_usa_structure(pdf_path: Path) -> list[OutlineEntry]:
    entries: list[OutlineEntry] = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            records = extract_line_records(page)
            i = 0
            while i < len(records):
                text = records[i]["text"]
                size = records[i]["max_size"]

                if re.fullmatch(r"Chapter\s+\d+", text):
                    if i + 1 < len(records) and records[i + 1]["max_size"] >= 15:
                        title = records[i + 1]["text"]
                        entries.append(
                            OutlineEntry(
                                title=f"{text}: {title}",
                                page=page_num,
                                depth=0,
                                source="synthetic_heading",
                            )
                        )
                        i += 2
                        continue

                if page_num <= 2 and re.match(r"^Chapter\s+\d+:", text):
                    i += 1
                    continue
                if re.match(r"^Chapter\s+\d+:", text):
                    i += 1
                    continue

                if size >= 17:
                    # Main title / intro page headings.
                    entries.append(
                        OutlineEntry(
                            title=text,
                            page=page_num,
                            depth=0 if text == "Introduction" else 1,
                            source="synthetic_heading",
                        )
                    )
                elif (
                    13.2 <= size < 17
                    and len(text.split()) <= 18
                    and not text.endswith((".", ";", ","))
                    and not text.startswith("AVG, Chapter ")
                ):
                    entries.append(
                        OutlineEntry(
                            title=text,
                            page=page_num,
                            depth=1,
                            source="synthetic_heading",
                        )
                    )
                elif (
                    11.8 <= size < 13.2
                    and len(text.split()) <= 14
                    and not text.endswith((".", ";", ","))
                    and text not in {"Special Rule", "PJ Examples", "Verification and PJ", "Verification items", "Acceptable documentation"}
                ):
                    entries.append(
                        OutlineEntry(
                            title=text,
                            page=page_num,
                            depth=2,
                            source="synthetic_heading",
                        )
                    )
                i += 1

    deduped: list[OutlineEntry] = []
    seen: set[tuple[int, int, str]] = set()
    for entry in entries:
        key = (entry.page, entry.depth, canonical_heading(entry.title))
        if key in seen:
            continue
        seen.add(key)
        deduped.append(entry)
    return deduped


def build_structure(pdf_path: Path, jurisdiction: str) -> list[OutlineEntry]:
    if jurisdiction == "Australia":
        return parse_australia_toc(pdf_path)
    if jurisdiction == "USA":
        return build_usa_structure(pdf_path)
    reader = PdfReader(str(pdf_path))
    entries = flatten_pdf_outline(reader)
    return sorted(entries, key=lambda e: (e.page, e.depth, e.title))


def path_for_entry(entries: list[OutlineEntry], target: OutlineEntry) -> list[str]:
    stack: list[OutlineEntry] = []
    for entry in entries:
        while stack and stack[-1].depth >= entry.depth:
            stack.pop()
        stack.append(entry)
        if entry is target:
            break
    return [s.title for s in stack]


def build_heading_lookup(entries: list[OutlineEntry]) -> dict[str, list[str]]:
    lookup: dict[str, list[str]] = {}
    stack: list[OutlineEntry] = []
    for entry in entries:
        while stack and stack[-1].depth >= entry.depth:
            stack.pop()
        stack.append(entry)
        lookup[canonical_heading(entry.title)] = [s.title for s in stack]
    return lookup


def default_path_for_page(entries: list[OutlineEntry], page: int) -> list[str]:
    prior = [entry for entry in entries if entry.page <= page]
    if not prior:
        return ["Front matter"]
    return path_for_entry(entries, prior[-1])


In [10]:
# =============================================================================
# Layout Extraction


In [11]:
# =============================================================================


def extract_line_records(page: pdfplumber.page.Page) -> list[dict]:
    words = page.extract_words(
        x_tolerance=2,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False,
        extra_attrs=["size", "fontname"],
    )
    if not words:
        raw = page.extract_text() or ""
        return [{"text": normalize_whitespace(x), "max_size": 0.0} for x in raw.splitlines() if normalize_whitespace(x)]

    buckets: dict[int, list[dict]] = defaultdict(list)
    for word in words:
        key = round(word["top"] / 3) * 3
        buckets[key].append(word)

    lines: list[dict] = []
    for key in sorted(buckets):
        row = sorted(buckets[key], key=lambda w: w["x0"])
        text = normalize_whitespace(" ".join(w["text"] for w in row))
        if not text:
            continue
        max_size = max(float(w["size"]) for w in row if w.get("size"))
        lines.append({"text": text, "max_size": max_size})
    return lines


def extract_lines(page: pdfplumber.page.Page) -> list[str]:
    return [record["text"] for record in extract_line_records(page)]


def extract_tables(page: pdfplumber.page.Page) -> list[list[list[str]]]:
    tables: list[list[list[str]]] = []
    try:
        for table in page.extract_tables() or []:
            cleaned: list[list[str]] = []
            for row in table:
                cleaned_row = [normalize_whitespace(cell or "") for cell in row]
                if any(cleaned_row):
                    cleaned.append(cleaned_row)
            if cleaned:
                tables.append(cleaned)
    except Exception:
        return []
    return tables


def table_to_markdown(table: list[list[str]]) -> str:
    width = max(len(row) for row in table)
    rows = [row + [""] * (width - len(row)) for row in table]
    header = rows[0]
    body = rows[1:] if len(rows) > 1 else []
    out = [
        "| " + " | ".join(header) + " |",
        "| " + " | ".join("---" for _ in header) + " |",
    ]
    for row in body:
        out.append("| " + " | ".join(row) + " |")
    return "\n".join(out)


In [12]:
# =============================================================================
# Heading, Paragraph, and Content-Type Detection


In [13]:
# =============================================================================


PARA_RE = re.compile(r"^((?:\d+\.)+\d*|\d+\.)\s+(.+)$")


def parse_numbered_line(line: str) -> tuple[str | None, str | None]:
    line = normalize_whitespace(line)
    m = PARA_RE.match(line)
    if not m:
        return None, None
    para = m.group(1)
    rest = m.group(2).strip()
    if re.match(rf"^(?:{MONTHS})\b", rest, re.I):
        return None, None
    return para, rest


def is_toc_line(line: str) -> bool:
    return bool(re.search(r"\.{4,}\s*\d+\s*$", line))


def is_update_heading(line: str) -> bool:
    return bool(re.match(r"^New for\s+\d{1,2}\s+" + MONTHS, line, re.I))


def is_update_note(line: str) -> bool:
    return bool(
        re.match(
            r"^(?:-?\s*)?(?:\d+(?:\.\d+)?(?:\s*/\s*\d+(?:\.\d+)?)?\s+)?(?:Clarification|Policy update):",
            line,
            re.I,
        )
    )


def strip_update_note_prefix(line: str) -> str:
    text = normalize_whitespace(line)
    text = re.sub(
        r"^(?:-?\s*)?(?:\d+(?:\.\d+)?(?:\s*/\s*\d+(?:\.\d+)?)?\s+)?(?:Clarification|Policy update):\s*",
        "",
        text,
        flags=re.I,
    )
    return normalize_whitespace(text)


def is_local_heading(line: str, jurisdiction: str) -> bool:
    l = normalize_whitespace(line)
    if not l or len(l) > 90:
        return False
    if l.endswith((".", ";", ",")):
        return False
    low = l.lower()
    if low in {"evidence requirements", "definitions", "overview", "policy", "purpose", "guidelines", "background"}:
        return True
    if jurisdiction == "UK" and low.startswith(("new for ", "assessment process", "claiming ")):
        return True
    if jurisdiction == "Canada" and low in {"purpose", "policy", "guidelines"}:
        return True
    if jurisdiction == "Australia" and re.match(r"^[A-Z][A-Za-z\s-]{3,60}$", l) and len(l.split()) <= 8:
        return True
    if jurisdiction == "USA":
        if re.match(r"^[A-Z][A-Za-z0-9®'()/:,\-\s]{3,90}$", l) and len(l.split()) <= 14:
            if low.startswith(("chapter ", "2025-26 ", "volume ", "electronic announcement ")):
                return False
            return True
    return False


def match_heading_at(lines: list[str], index: int, lookup: dict[str, list[str]]) -> tuple[list[str] | None, int]:
    max_join = min(3, len(lines) - index)
    for n in range(max_join, 0, -1):
        combined = normalize_whitespace(" ".join(lines[index : index + n]))
        key = canonical_heading(combined)
        if key in lookup:
            return lookup[key], n
    return None, 0


def classify_content_type(
    text: str,
    section_path: list[str],
    local_heading: str | None,
    paragraph_number: str | None,
    jurisdiction: str,
) -> str:
    low = text.lower()
    path_low = " > ".join(section_path).lower()
    local_low = (local_heading or "").lower()

    if is_toc_line(text) or path_low.startswith("front matter"):
        return "toc" if is_toc_line(text) else "front_matter"
    if is_update_heading(text) or is_update_note(text) or "key updates" in path_low:
        return "update_note"
    if "glossary" in path_low or "definitions" in path_low or local_low == "definitions":
        return "definition"
    if jurisdiction == "USA" and (local_low.endswith("example") or "example" in local_low):
        return "example"
    if jurisdiction == "USA" and ("34 cfr" in text.lower() or "hea sec." in text.lower()):
        return "citation"
    if "table" in path_low or text.lstrip().startswith("|"):
        return "table"
    if "contact" in path_low:
        return "contact"
    if local_low == "evidence requirements" or low.startswith("evidence requirements"):
        return "evidence_requirement"
    if paragraph_number:
        return "numbered_rule"
    if re.match(r"^[•\-o]\s+", text):
        return "list_item"
    return "paragraph"


def extract_thresholds(text: str) -> list[str]:
    patterns = [
        r"(?:£|\$|AUD\s*|CAD\s*)\s?\d[\d,]*(?:\.\d+)?",
        r"\b\d+(?:\.\d+)?\s?%",
        r"\b(?:under|over|aged?|age of|at least)\s+\d{1,2}\b",
        r"\b\d+\s+(?:business\s+days?|working\s+days?|day|days|week|weeks|month|months|year|years|hour|hours)\b",
        r"\b(?:at least|no more than|minimum of|maximum of|less than|more than)\s+\d+\b",
    ]
    values: list[str] = []
    for pat in patterns:
        values.extend(re.findall(pat, text, flags=re.I))
    return list(dict.fromkeys(normalize_whitespace(v) for v in values if normalize_whitespace(v)))


def extract_temporal_cues(text: str) -> list[str]:
    patterns = [
        r"\bwithin\s+\d+\s+(?:business\s+days?|working\s+days?|day|days|week|weeks|month|months|year|years)\b",
        r"\bno later than\b[^.,;]*",
        r"\bwith effect from\b[^.,;]*",
        r"\beffective from\b[^.,;]*",
        r"\bon or after\s+\d{1,2}\s+(?:" + MONTHS + r")\s+\d{4}\b",
        r"\bby\s+\d{1,2}\s+(?:" + MONTHS + r")\s+\d{4}\b",
        r"\b20\d{2}[\u2013\-]\d{2,4}\b",
    ]
    cues: list[str] = []
    for pat in patterns:
        cues.extend(re.findall(pat, text, flags=re.I))
    return list(dict.fromkeys(normalize_whitespace(c) for c in cues if normalize_whitespace(c)))[:8]


def split_long_text(text: str, max_words: int = 150) -> list[str]:
    """Split long prose/list chunks after extraction, before provision scoring."""
    text = normalize_whitespace(text)
    if len(text.split()) <= max_words:
        return [text]

    def hard_split(value: str) -> list[str]:
        words = value.split()
        return [" ".join(words[start : start + max_words]) for start in range(0, len(words), max_words)]

    def enforce_max(values: list[str]) -> list[str]:
        out: list[str] = []
        for value in values:
            if len(value.split()) <= max_words:
                out.append(value)
            else:
                out.extend(hard_split(value))
        return [seg for seg in out if len(seg.split()) >= 5]

    if text.startswith("| "):
        lines = text.splitlines()
        header = lines[:2] if len(lines) >= 2 else []
        body = lines[2:] if len(lines) >= 2 else lines
        segments: list[str] = []
        current = header[:]
        current_words = sum(len(line.split()) for line in current)
        for row in body:
            row_words = len(row.split())
            if current and current_words + row_words > max_words and len(current) > len(header):
                segments.append("\n".join(current))
                current = header[:] + [row]
                current_words = sum(len(line.split()) for line in current)
            else:
                current.append(row)
                current_words += row_words
        if current:
            segments.append("\n".join(current))
        return enforce_max(segments)

    if "•" in text:
        parts = [normalize_whitespace(p) for p in re.split(r"\s*•\s+", text) if normalize_whitespace(p)]
        if len(parts) > 1:
            lead = parts[0]
            if len(lead.split()) < 25:
                return enforce_max([f"• {p}" for p in parts[1:]])
            return enforce_max([lead] + [f"• {p}" for p in parts[1:]])

    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z(•])", text)
    if len(sentences) <= 1:
        return [seg for seg in hard_split(text) if len(seg.split()) >= 5]

    segments: list[str] = []
    current: list[str] = []
    current_words = 0
    for sentence in sentences:
        wc = len(sentence.split())
        if current and current_words + wc > max_words:
            segments.append(normalize_whitespace(" ".join(current)))
            current = [sentence]
            current_words = wc
        else:
            current.append(sentence)
            current_words += wc
    if current:
        segments.append(normalize_whitespace(" ".join(current)))
    final_segments: list[str] = []
    for seg in segments:
        words = seg.split()
        if len(words) <= max_words:
            final_segments.append(seg)
            continue
        final_segments.extend(hard_split(seg))
    return enforce_max(final_segments)


def split_definition_terms(text: str) -> list[str]:
    """Split dense glossary/definition blocks into individual terms where possible."""
    text = normalize_whitespace(text)
    if len(text.split()) < 20:
        return [text]
    intro = ""
    if text.startswith("This chapter lists the key definitions used in this manual."):
        intro = "This chapter lists the key definitions used in this manual."
        text = normalize_whitespace(text[len(intro) :])

    term_word = r"(?:[A-Z][A-Za-z0-9&'()/\-]+|[A-Z]{2,}|\([A-Z]{2,}\)|of|and|for|to|the|or|with|in)"
    starter = r"(?:A|An|The|This|These|Where|When|Any|All|Extra|Loans|Funding|Used|Means|Refers|Is|Are|See)\b"
    pattern = re.compile(
        rf"(?P<term>{term_word}(?:\s+{term_word}){{0,7}})\s+"
        rf"(?P<definition>{starter}.*?)(?=(?:{term_word}(?:\s+{term_word}){{0,7}}\s+{starter})|$)"
    )

    parts: list[str] = []
    pos = 0
    for match in pattern.finditer(text):
        if match.start() > pos:
            leftover = normalize_whitespace(text[pos : match.start()])
            if len(leftover.split()) >= 4:
                parts.append(leftover)
        part = normalize_whitespace(f"{match.group('term')} {match.group('definition')}")
        if len(part.split()) >= 4:
            parts.append(part)
        pos = match.end()

    if pos < len(text):
        tail = normalize_whitespace(text[pos:])
        if len(tail.split()) >= 4:
            parts.append(tail)

    if intro:
        parts = [intro] + parts

    if len(parts) >= 2:
        return parts
    return [normalize_whitespace((intro + " " + text).strip())]


In [14]:
# =============================================================================
# Chunk Extraction


In [15]:
# =============================================================================


def should_flush_on_line(line: str) -> bool:
    if not line:
        return True
    if is_toc_line(line):
        return True
    return False


def begins_new_block(line: str) -> bool:
    if parse_numbered_line(line)[0]:
        return True
    if re.match(r"^[•\-o]\s+", line):
        return True
    return False


def should_keep_clean_chunk(chunk: Chunk) -> bool:
    """
    LLM-ready clean chunk filter: keep semantically meaningful content only.
    """
    text = normalize_whitespace(chunk.text)
    wc = len(text.split())
    if not text:
        return False
    if is_noise_text(text) or is_garbled_text(text):
        return False
    if chunk.content_type in {"front_matter", "toc", "update_note", "contact", "example", "citation"}:
        return False
    if chunk.local_heading and chunk.local_heading.lower().startswith("new for ") and wc < 12:
        return False
    # Definitions: always keep if wc >= 4 — do NOT apply local_heading short-text
    # guard. Legal definitions are standalone semantic units.
    if chunk.content_type == "definition":
        return wc >= 4
    if chunk.content_type == "table":
        return wc >= 4
    if chunk.paragraph_number:
        return wc >= 4
    if chunk.local_heading and wc < 12:
        return False
    return wc >= 8


def make_source_anchor(
    jurisdiction: str,
    pdf_page: int,
    section_path: list[str],
    paragraph_number: str | None,
    local_heading: str | None,
    chunk_order: int | None = None,
) -> str:
    sec = " > ".join(section_path[-3:]) if section_path else "General"
    parts = [jurisdiction, f"p{pdf_page}", sec]
    if local_heading:
        parts.append(local_heading)
    if paragraph_number:
        parts.append(paragraph_number)
    if chunk_order is not None:
        parts.append(f"ord-{chunk_order:04d}")
    raw = " | ".join(parts)
    return re.sub(r"\s+", " ", raw)


def extract_chunks(pdf_path: Path, jurisdiction: str, entries: list[OutlineEntry]) -> list[Chunk]:
    chunks: list[Chunk] = []
    heading_lookup = build_heading_lookup(entries)
    chunk_order = 0

    current_section_path = ["Front matter"]
    current_local_heading: str | None = None
    block: list[str] = []
    block_para: str | None = None
    block_page = 1
    block_printed: str | None = None

    def flush() -> None:
        nonlocal block, block_para, block_page, block_printed, chunk_order
        if not block:
            return
        text = clean_inline_furniture(normalize_whitespace(" ".join(block)), jurisdiction)
        block = []
        if not text or len(text.split()) < 3:
            block_para = None
            return
        chunk_order += 1
        content_type = classify_content_type(
            text=text,
            section_path=current_section_path,
            local_heading=current_local_heading,
            paragraph_number=block_para,
            jurisdiction=jurisdiction,
        )
        chunks.append(
            Chunk(
                jurisdiction=jurisdiction,
                source_file=pdf_path.name,
                pdf_page=block_page,
                printed_page=block_printed,
                section_path=current_section_path[:],
                local_heading=current_local_heading,
                paragraph_number=block_para,
                content_type=content_type,
                text=text,
                source_anchor=make_source_anchor(
                    jurisdiction, block_page, current_section_path, block_para, current_local_heading, chunk_order
                ),
                thresholds=extract_thresholds(text),
                temporal_cues=extract_temporal_cues(text),
            )
        )
        block_para = None

    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            raw_lines = extract_lines(page)
            printed_page = printed_page_from_lines(raw_lines, jurisdiction, page_num)
            lines = [line for line in raw_lines if not is_page_furniture(line, jurisdiction)]
            if not lines:
                continue

            page_default_path = default_path_for_page(entries, page_num)
            if page_default_path:
                current_section_path = page_default_path
                current_local_heading = None

            tables = extract_tables(page)
            for table_idx, table in enumerate(tables, start=1):
                table_text = table_to_markdown(table)
                if len(table_text.split()) < 4:
                    continue
                chunk_order += 1
                chunks.append(
                    Chunk(
                        jurisdiction=jurisdiction,
                        source_file=pdf_path.name,
                        pdf_page=page_num,
                        printed_page=printed_page,
                        section_path=page_default_path,
                        local_heading="Table",
                        paragraph_number=None,
                        content_type="table",
                        text=table_text,
                        source_anchor=make_source_anchor(
                            jurisdiction, page_num, page_default_path, None, f"table-{table_idx}", chunk_order
                        ),
                        thresholds=extract_thresholds(table_text),
                        temporal_cues=extract_temporal_cues(table_text),
                    )
                )

            i = 0
            while i < len(lines):
                line = clean_inline_furniture(lines[i], jurisdiction)
                if not line:
                    i += 1
                    continue

                matched_path, consumed = match_heading_at(lines, i, heading_lookup)
                if matched_path:
                    current_path_text = " > ".join(current_section_path).lower()
                    matched_path_text = " > ".join(matched_path).lower()
                    if (
                        ("glossary" in current_path_text and "glossary" not in matched_path_text)
                        or ("definitions" in current_path_text and "definitions" not in matched_path_text)
                    ):
                        matched_path = None
                    else:
                        flush()
                        current_section_path = matched_path
                        current_local_heading = None
                        i += consumed
                        continue

                if matched_path:
                    flush()
                    current_section_path = matched_path
                    current_local_heading = None
                    i += consumed
                    continue

                if is_toc_line(line):
                    flush()
                    chunk_order += 1
                    chunks.append(
                        Chunk(
                            jurisdiction=jurisdiction,
                            source_file=pdf_path.name,
                            pdf_page=page_num,
                            printed_page=printed_page,
                            section_path=["Front matter"],
                            local_heading="Table of contents",
                            paragraph_number=None,
                            content_type="toc",
                            text=line,
                            source_anchor=make_source_anchor(
                                jurisdiction, page_num, ["Front matter"], None, "toc", chunk_order
                            ),
                        )
                    )
                    i += 1
                    continue

                if is_update_heading(line) or is_update_note(line):
                    # Keep update headings in debug output, but avoid fragmenting the
                    # following explanatory text into tiny chunks.
                    short_block = normalize_whitespace(" ".join(block))
                    if (
                        block
                        and current_local_heading
                        and current_local_heading.lower().startswith("new for ")
                        and len(short_block.split()) < 12
                    ):
                        current_local_heading = current_local_heading
                        i += 1
                        continue

                    flush()
                    if is_update_heading(line):
                        current_local_heading = line
                        chunk_order += 1
                        chunks.append(
                            Chunk(
                                jurisdiction=jurisdiction,
                                source_file=pdf_path.name,
                                pdf_page=page_num,
                                printed_page=printed_page,
                                section_path=current_section_path[:],
                                local_heading="Update note",
                                paragraph_number=None,
                                content_type="update_note",
                                text=line,
                                source_anchor=make_source_anchor(
                                    jurisdiction, page_num, current_section_path, None, "update-note", chunk_order
                                ),
                                thresholds=extract_thresholds(line),
                                temporal_cues=extract_temporal_cues(line),
                            )
                        )
                        i += 1
                        continue

                    stripped_note = strip_update_note_prefix(line)
                    chunk_order += 1
                    chunks.append(
                        Chunk(
                            jurisdiction=jurisdiction,
                            source_file=pdf_path.name,
                            pdf_page=page_num,
                            printed_page=printed_page,
                            section_path=current_section_path[:],
                            local_heading="Update note",
                            paragraph_number=None,
                            content_type="update_note",
                            text=line,
                            source_anchor=make_source_anchor(
                                jurisdiction, page_num, current_section_path, None, "update-note", chunk_order
                            ),
                            thresholds=extract_thresholds(line),
                            temporal_cues=extract_temporal_cues(line),
                        )
                    )
                    if stripped_note:
                        block = [stripped_note]
                        block_para = None
                        block_page = page_num
                        block_printed = printed_page
                        current_local_heading = current_local_heading or "Update note"
                    i += 1
                    continue

                if is_local_heading(line, jurisdiction):
                    flush()
                    current_local_heading = line
                    i += 1
                    continue

                para, rest = parse_numbered_line(line)
                if para:
                    flush()
                    block = [rest] if rest else []
                    block_para = para
                    block_page = page_num
                    block_printed = printed_page
                    i += 1
                    continue

                if begins_new_block(line):
                    flush()
                    block = [line]
                    block_para = None
                    block_page = page_num
                    block_printed = printed_page
                    i += 1
                    continue

                if not block:
                    block_page = page_num
                    block_printed = printed_page
                block.append(line)
                i += 1

            flush()

    return chunks


In [16]:
# =============================================================================
# Output Building


In [17]:
# =============================================================================


def stable_slug(text: str, max_len: int = 64) -> str:
    slug = canonical_heading(text)
    slug = re.sub(r"\s+", "-", slug)
    return slug[:max_len].strip("-") or "chunk"


def is_contextual_narrative(chunk: Chunk) -> bool:
    return chunk.content_type in {"paragraph", "numbered_rule", "definition", "evidence_requirement", "list_item"}


def chunk_to_row(chunk: Chunk, chunk_id: str) -> dict:
    row = asdict(chunk)
    row["chunk_id"] = chunk_id
    row["section"] = " > ".join(chunk.section_path)
    row["section_id"] = stable_slug(row["section"])
    row["word_count"] = len(chunk.text.split())
    return row


def build_prepared_units(clean_chunks: list[Chunk], clean_rows: list[dict], jurisdiction: str, pdf_path: Path) -> list[PreparedUnit]:
    units: list[PreparedUnit] = []
    row_by_id = {row["chunk_id"]: row for row in clean_rows}
    seen_same_section: dict[tuple[str, str], int] = {}

    def needs_parent_context(chunk: Chunk) -> bool:
        text = normalize_whitespace(chunk.text)
        wc = len(text.split())
        if wc == 0:
            return False
        if chunk.content_type == "list_item" and wc <= 12:
            return True
        if chunk.content_type in {"numbered_rule", "paragraph"} and wc <= 8:
            return True
        if text.endswith((";", ":", ",", "and", "or")):
            return True
        if re.search(r"\b(?:you must:|including:|following:)$", text, re.I):
            return True
        if re.match(r"^Formula [A-Z0-9]+\b", text):
            return True
        return False

    def find_parent_support(idx: int) -> list[str]:
        chunk = clean_chunks[idx]
        accepted = {"paragraph", "numbered_rule", "definition", "evidence_requirement", "list_item"}
        parent_path = chunk.section_path[:-1]

        def scan(indices) -> list[str]:
            for j in indices:
                other_chunk = clean_chunks[j]
                other_row = clean_rows[j]
                if abs(other_chunk.pdf_page - chunk.pdf_page) > 1:
                    break
                same_section = other_chunk.section_path == chunk.section_path
                same_parent = parent_path and other_chunk.section_path[:-1] == parent_path
                if not same_section and not same_parent:
                    break
                if other_chunk.content_type == "table":
                    continue
                if other_chunk.content_type in accepted and normalize_whitespace(other_chunk.text) != normalize_whitespace(chunk.text):
                    return [other_row["chunk_id"]]
            return []

        backward = scan(range(idx - 1, -1, -1))
        if backward:
            return backward
        forward = scan(range(idx + 1, len(clean_chunks)))
        if forward:
            return forward

        # Final fallback for very short heading-like fragments that often sit on
        # subsection boundaries: allow the nearest accepted neighbor within a
        # tight local window if it is on the same or adjacent page.
        for distance in range(1, 6):
            for j in (idx - distance, idx + distance):
                if j < 0 or j >= len(clean_chunks):
                    continue
                other_chunk = clean_chunks[j]
                other_row = clean_rows[j]
                if other_chunk.content_type not in accepted:
                    continue
                if abs(other_chunk.pdf_page - chunk.pdf_page) > 1:
                    continue
                if normalize_whitespace(other_chunk.text) == normalize_whitespace(chunk.text):
                    continue
                return [other_row["chunk_id"]]
        return []

    def find_table_support(idx: int) -> tuple[list[dict], list[dict]]:
        chunk = clean_chunks[idx]
        support_before: list[dict] = []
        support_after: list[dict] = []

        j = idx - 1
        while j >= 0 and len(support_before) < 2:
            other = clean_chunks[j]
            other_row = clean_rows[j]
            if other.content_type == "table":
                break
            if other.section_path != chunk.section_path or abs(other.pdf_page - chunk.pdf_page) > 1:
                break
            if is_contextual_narrative(other):
                support_before.insert(0, other_row)
            j -= 1

        j = idx + 1
        while j < len(clean_chunks) and len(support_after) < 2:
            other = clean_chunks[j]
            other_row = clean_rows[j]
            if other.content_type == "table":
                break
            if other.section_path != chunk.section_path or abs(other.pdf_page - chunk.pdf_page) > 1:
                break
            if is_contextual_narrative(other):
                support_after.append(other_row)
            j += 1

        return support_before, support_after

    def append_unit(
        *,
        unit_type: str,
        pdf_page_start: int,
        pdf_page_end: int,
        printed_page_start: str | None,
        printed_page_end: str | None,
        section_path: list[str],
        local_heading: str | None,
        paragraph_number: str | None,
        source_anchor: str,
        text: str,
        component_chunk_ids: list[str],
        support_chunk_ids: list[str] | None = None,
    ) -> None:
        parts = split_long_text(text, max_words=220)
        if not parts:
            parts = [normalize_whitespace(text)]
        for part_idx, part in enumerate(parts, start=1):
            part = normalize_whitespace(part)
            if not part or is_noise_text(part) or is_garbled_text(part):
                continue
            part_support_ids = (support_chunk_ids or [])[:]
            if len(parts) > 1 and len(part.split()) <= 12 and not part_support_ids:
                part_support_ids = component_chunk_ids[:1]
            # Deduplication: only suppress exact duplicates within the SAME
            # section. Cross-section duplicates are kept — same rule appearing
            # in two different sections (e.g., main body + Annex B) carries
            # different provenance and should be retrievable from both contexts.
            dedup_key = (
                stable_slug(" > ".join(section_path)),
                _content_fingerprint(part),
            )
            existing_idx = seen_same_section.get(dedup_key)
            if existing_idx is not None:
                existing = units[existing_idx]
                prefer_new = (
                    unit_type == "table_bundle" and existing.unit_type != "table_bundle"
                )
                if not prefer_new:
                    continue
                units[existing_idx] = PreparedUnit(
                    jurisdiction=jurisdiction,
                    source_file=pdf_path.name,
                    unit_id=existing.unit_id,
                    unit_type=unit_type,
                    pdf_page_start=pdf_page_start,
                    pdf_page_end=pdf_page_end,
                    printed_page_start=printed_page_start,
                    printed_page_end=printed_page_end,
                    section_path=section_path[:],
                    section=" > ".join(section_path),
                    section_id=stable_slug(" > ".join(section_path)),
                    local_heading=local_heading,
                    paragraph_number=paragraph_number,
                    source_anchor=source_anchor if len(parts) == 1 else f"{source_anchor} | part-{part_idx}",
                    text=part,
                    component_chunk_ids=component_chunk_ids[:],
                    support_chunk_ids=part_support_ids or (component_chunk_ids[:1] if len(part.split()) <= 6 else []),
                    thresholds=extract_thresholds(part),
                    temporal_cues=extract_temporal_cues(part),
                )
                continue
            units.append(
                PreparedUnit(
                    jurisdiction=jurisdiction,
                    source_file=pdf_path.name,
                    unit_id=f"{jurisdiction[:3].upper()}-UNIT-{len(units) + 1:05d}",
                    unit_type=unit_type,
                    pdf_page_start=pdf_page_start,
                    pdf_page_end=pdf_page_end,
                    printed_page_start=printed_page_start,
                    printed_page_end=printed_page_end,
                    section_path=section_path[:],
                    section=" > ".join(section_path),
                    section_id=stable_slug(" > ".join(section_path)),
                    local_heading=local_heading,
                    paragraph_number=paragraph_number,
                    source_anchor=source_anchor if len(parts) == 1 else f"{source_anchor} | part-{part_idx}",
                    text=part,
                    component_chunk_ids=component_chunk_ids[:],
                    support_chunk_ids=(support_chunk_ids or [])[:],
                    thresholds=extract_thresholds(part),
                    temporal_cues=extract_temporal_cues(part),
                )
            )
            seen_same_section[dedup_key] = len(units) - 1


    absorbed_chunk_ids: set[str] = set()
    table_bundle_plans: list[dict] = []

    for idx, chunk in enumerate(clean_chunks):
        if chunk.content_type != "table":
            continue
        table_row = clean_rows[idx]
        support_before, support_after = find_table_support(idx)
        if not (support_before or support_after):
            continue

        component_rows = support_before + [table_row] + support_after
        table_bundle_plans.append({
            "chunk": chunk,
            "component_rows": component_rows,
            "support_rows": support_before + support_after,
        })
        for r in component_rows:
            absorbed_chunk_ids.add(r["chunk_id"])

    # ---- PASS 2: emit standalone units for chunks not absorbed into a bundle.
    # O(n) iteration via enumerate (previous O(n^2) clean_rows.index removed).
    for row_idx, (row, chunk) in enumerate(zip(clean_rows, clean_chunks)):
        if row["chunk_id"] in absorbed_chunk_ids:
            continue
        support_ids: list[str] = []
        if needs_parent_context(chunk):
            support_ids = find_parent_support(row_idx)
        append_unit(
            unit_type=chunk.content_type,
            pdf_page_start=chunk.pdf_page,
            pdf_page_end=chunk.pdf_page,
            printed_page_start=chunk.printed_page,
            printed_page_end=chunk.printed_page,
            section_path=chunk.section_path,
            local_heading=chunk.local_heading,
            paragraph_number=chunk.paragraph_number,
            source_anchor=chunk.source_anchor,
            text=chunk.text,
            component_chunk_ids=[row["chunk_id"]],
            support_chunk_ids=support_ids,
        )

    # ---- PASS 3: emit table_bundle units.
    for plan in table_bundle_plans:
        chunk = plan["chunk"]
        component_rows = plan["component_rows"]
        support_rows = plan["support_rows"]
        bundle_text = "\n\n".join(r["text"] for r in component_rows)
        append_unit(
            unit_type="table_bundle",
            pdf_page_start=min(r["pdf_page"] for r in component_rows),
            pdf_page_end=max(r["pdf_page"] for r in component_rows),
            printed_page_start=component_rows[0]["printed_page"],
            printed_page_end=component_rows[-1]["printed_page"],
            section_path=chunk.section_path,
            local_heading=chunk.local_heading or "Table bundle",
            paragraph_number=None,
            source_anchor=f"{chunk.source_anchor} | table-bundle",
            text=bundle_text,
            component_chunk_ids=[r["chunk_id"] for r in component_rows],
            support_chunk_ids=[r["chunk_id"] for r in support_rows],
        )

    # ---- POST-PROCESS: short stub units get self-support if otherwise unsupported.
    for unit in units:
        if not unit.support_chunk_ids and len(unit.text.split()) <= 6 and unit.component_chunk_ids:
            unit.support_chunk_ids = unit.component_chunk_ids[:1]

    return units


def build_structure_doc(jurisdiction: str, pdf_path: Path, entries: list[OutlineEntry]) -> dict:
    rows = []
    for idx, entry in enumerate(entries, start=1):
        rows.append(
            {
                "outline_id": f"{jurisdiction[:3].upper()}-OUTLINE-{idx:04d}",
                "jurisdiction": jurisdiction,
                "source_file": pdf_path.name,
                "title": entry.title,
                "page": entry.page,
                "depth": entry.depth,
                "source": entry.source,
                "printed_page": entry.printed_page,
                "title_slug": stable_slug(entry.title),
            }
        )
    return {
        "metadata": {
            "jurisdiction": jurisdiction,
            "source_document": pdf_path.name,
            "pipeline_layer": "Layer 1 - Document Preparation",
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "total_outline_entries": len(rows),
        },
        "structure_map": rows,
    }


def build_preparation_outputs(
    jurisdiction: str,
    pdf_path: Path,
    entries: list[OutlineEntry],
    chunks: list[Chunk],
) -> tuple[dict, dict, dict, dict]:
    debug_chunk_rows = [chunk_to_row(chunk, f"{jurisdiction[:3].upper()}-CHUNK-{idx:05d}") for idx, chunk in enumerate(chunks, start=1)]

    clean_chunks = [chunk for chunk in chunks if should_keep_clean_chunk(chunk)]
    clean_chunk_rows = [chunk_to_row(chunk, f"{jurisdiction[:3].upper()}-CLEAN-{idx:05d}") for idx, chunk in enumerate(clean_chunks, start=1)]

    final_chunks = build_prepared_units(clean_chunks, clean_chunk_rows, jurisdiction, pdf_path)
    final_chunk_rows = [asdict(unit) for unit in final_chunks]

    metadata = {
        "jurisdiction": jurisdiction,
        "source_document": pdf_path.name,
        "pipeline_layer": "Layer 1 - Document Preparation",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "total_outline_entries": len(entries),
        "total_chunks_debug": len(debug_chunk_rows),
        "total_chunks_clean": len(clean_chunk_rows),
        "total_final_chunks": len(final_chunk_rows),
    }

    return (
        build_structure_doc(jurisdiction, pdf_path, entries),
        {"metadata": metadata, "prepared_chunks_debug": debug_chunk_rows},
        {"metadata": metadata, "prepared_chunks_clean": clean_chunk_rows},
        {"metadata": metadata, "final_chunks": final_chunk_rows},
    )


def quality_report(jurisdiction: str, structure_doc: dict, debug_doc: dict, clean_doc: dict, final_doc: dict) -> None:
    chunks_debug = debug_doc["prepared_chunks_debug"]
    chunks_clean = clean_doc["prepared_chunks_clean"]
    final_chunks = final_doc["final_chunks"]

    print(f"\n{jurisdiction}")
    print(f"  outline      : {len(structure_doc['structure_map'])}")
    print(f"  chunks debug : {len(chunks_debug)}")
    print(f"  chunks clean : {len(chunks_clean)}")
    print(f"  final chunks : {len(final_chunks)}")
    if final_chunks:
        word_counts = [len(p["text"].split()) for p in final_chunks]
        print(f"  avg words    : {mean(word_counts):.1f}")
        print(f"  max words    : {max(word_counts)}")
        print(f"  unit types   : {Counter(p['unit_type'] for p in final_chunks).most_common(8)}")
        print(f"  top sections : {Counter(p['section'] for p in final_chunks).most_common(5)}")


def main() -> None:
    for jurisdiction, pdf_path in DOCUMENTS.items():
        if not pdf_path.exists():
            raise FileNotFoundError(f"{jurisdiction}: {pdf_path}")

        print(f"Processing {jurisdiction}: {pdf_path.name}")
        structure = build_structure(pdf_path, jurisdiction)
        chunks = extract_chunks(pdf_path, jurisdiction, structure)
        structure_doc, debug_doc, clean_doc, final_doc = build_preparation_outputs(
            jurisdiction, pdf_path, structure, chunks
        )

        structure_path = OUTPUT_DIR / f"structure_map_{jurisdiction.lower()}_v1.json"
        debug_chunks_path = OUTPUT_DIR / f"prepared_chunks_debug_{jurisdiction.lower()}_v1.json"
        clean_chunks_path = OUTPUT_DIR / f"prepared_chunks_clean_{jurisdiction.lower()}_v1.json"
        final_chunks_path = OUTPUT_DIR / f"final_chunks_{jurisdiction.lower()}_v1.json"

        structure_path.write_text(json.dumps(structure_doc, ensure_ascii=False, indent=2), encoding="utf-8")
        debug_chunks_path.write_text(json.dumps(debug_doc, ensure_ascii=False, indent=2), encoding="utf-8")
        clean_chunks_path.write_text(json.dumps(clean_doc, ensure_ascii=False, indent=2), encoding="utf-8")
        final_chunks_path.write_text(json.dumps(final_doc, ensure_ascii=False, indent=2), encoding="utf-8")

        quality_report(jurisdiction, structure_doc, debug_doc, clean_doc, final_doc)
        print(f"  wrote map    : {structure_path}")
        print(f"  wrote debug  : {debug_chunks_path}")
        print(f"  wrote clean  : {clean_chunks_path}")
        print(f"  wrote final  : {final_chunks_path}")


if __name__ == "__main__":
    main()



Processing UK: Funding_Rules_(uk).pdf

UK
  outline      : 62
  chunks debug : 1311
  chunks clean : 1226
  final chunks : 1225
  avg words    : 37.2
  max words    : 218
  unit types   : [('numbered_rule', 998), ('paragraph', 112), ('list_item', 64), ('definition', 46), ('table_bundle', 5)]
  top sections : [('Annex A: Residency eligibility criteria (who we fund)', 150), ('Delivery models > Subcontracting', 98), ('Evidence requirements', 85), ('Initial assessment > Support for English and maths training', 65), ('Annex B: Revisions to apprenticeship assessment', 40)]
  wrote map    : output/preparation_v1/structure_map_uk_v1.json
  wrote debug  : output/preparation_v1/prepared_chunks_debug_uk_v1.json
  wrote clean  : output/preparation_v1/prepared_chunks_clean_uk_v1.json
  wrote final  : output/preparation_v1/final_chunks_uk_v1.json
Processing Canada: policy_manual(canada).pdf

Canada
  outline      : 235
  chunks debug : 846
  chunks clean : 758
  final chunks : 886
  avg words    : 7